Excerise
1. Working with python package
2. Naive bayes (Gaussian, Bernoulli, Multinomial), KNN (KD Tree, Ball Tree)
3. Linear regression (Lasso, Ridge, Elastic Net)
4. Logistic Regression, SVM (Linear, Polynomial, RBF)
5. Neural Networks
6. Decision Tree, Random Forest
7. Bagging and boosting
8. PCA
9. Clustering (K Means, DBScan, Hierarchical)

###**Steps involved in the machine learning workflow**

1. Load Data
2. EDA and Visualization
3. Preprocessing
4. Feature Selection (Correlation / ANOVA)
5. Train-Test Split
6. Scaling -> Scaling is done after splitting to prevent data leakage, ensuring the model does not learn from test data statistics like mean and standard deviation.
7. Train Model
8. (Optional) PCA
9. Cross Validation
10. Hyperparameter Tuning (Grid/Random Search)
11. Evaluate (Accuracy, Precision / Recall / F1, ROC)
12. Compare Models (statistical test. ex: One way anova, Friedman Test)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# SKLEARN CORE
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.compose import *
from sklearn.impute import *
from sklearn.preprocessing import *

# MODELS
from sklearn.linear_model import *
from sklearn.svm import *
from sklearn.neighbors import *
from sklearn.naive_bayes import *
from sklearn.tree import *
from sklearn.ensemble import *
from sklearn.neural_network import *

# MODELS - DIMENSIONALITY REDUCTION & CLUSTERING (NEW)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
import scipy.cluster.hierarchy as shc

# METRICS
from sklearn.metrics import *
import time

In [ ]:
df=pd.read_csv('filename.csv')
print(df.head())

In [ ]:
# Basic Structure
print(df.head())
print(df.tail())
print(df.shape)
print(df.info())
print(df.describe(include='all',percentiles=[.01, .05, .95, .99]))

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(exclude=np.number).columns

## **Exploratory Data Analysis and Visualization**

In [ ]:
sns.pairplot(df[num_cols])
plt.show()

# Histogram of a numeric feature for skewness or gaussian distribution
sns.histplot(df['Income (USD)'], kde=True)
plt.show()

# outlier detection
sns.boxplot(x=df['target'], y=df['Income (USD)'])
plt.show()


# Correlation Matrix
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm')
plt.show()

# categorical column
# Class Distribution (Number of samples per class)
sns.countplot(x=df['target'])
plt.show()

# categorical column
sns.barplot(x=df['Gender'], y=df['Income (USD)'])
plt.show()

### Plotting Strategy

In [ ]:
#If we have two values (X, Y) → use scatter plot ex: Actual vs Predicted
import matplotlib.pyplot as plt
plt.scatter(X, Y)
plt.show()
# when Data is ordered (time / sequence) → use Line plot ex: Epoch vs Loss
plt.plot(X, Y)
plt.show()
# Use when:we have X = categories, Y = values → use Bar plot ex: Subjects vs Marks
plt.bar(X, Y)
plt.show()

## **Data Preprocessing**

1. Handle Missing Values

In [ ]:
#common strategy:{mean->numerical's,meadian->numerical's(robust to outliers),most_freuent->categorical}
#In pandas, isnull() and isna() are the same
print(df.isnull().sum()) # Count missing values
print(df.isnull().mean()) # Percentage of missing values


#1. Fill (Impute) missing values
from sklearn.impute import SimpleImputer
df[num_cols] = SimpleImputer(strategy="mean").fit_transform(df[num_cols]) # or "median", "most_frequent"
df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])


# Fill missing values in numerical columns (using median)
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing values in categorical columns (using mode)
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

#2. Drop missing values
# 1.1:Drop Rows
df.dropna(axis=0,inplace=True)
# 1.2:Drop Columns
df.dropna(axis=1, inplace=True)

2. Remove Duplicates (optional)

In [ ]:
print(df.duplicated().sum())  # Duplicate rows
print(df.nunique())           # Unique values per column
#remove duplicates
df.drop_duplicates(inplace=True)

3. Handle Outliers

In [ ]:
# Outlier detection/removal is applied only to numerical columns
import numpy as np

# Step 1: Select numeric columns
num_cols = df.select_dtypes(include=np.number).columns

# Step 2: Calculate IQR
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

# Step 3: Remove outliers and update df
l = Q1 - 1.5 * IQR
u = Q3 + 1.5 * IQR
o = ((df[num_cols] < l) | (df[num_cols] > u)).any(axis=1)
df = df[~o]

print("After IQR outlier removal:", df.shape)

4. Encode Categorical Data

In [ ]:
# Example split
nominal_cols = ['color']
ordinal_cols = ['size']

# One-hot
import pandas as pd
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

# Ordinal
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])
df[ordinal_cols] = oe.fit_transform(df[ordinal_cols])

# only for target varibales lable ex: "target": ["Buy", "Not Buy"]
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["target"] = le.fit_transform(df["target"])

##  **Feature Selection**

**Correlation-based (Simple & Fast)**

**Statistical Feature Selection**

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
# 1
corr = df.select_dtypes(include="number").corr()
high_corr = corr[target_col].drop(target_col).sort_values(ascending=False)
print(high_corr)

# 2
#Classification (target = categories / labels)
#ANOVA is used inside SelectKBest via f_classif .
from sklearn.feature_selection import SelectKBest, f_classif
selector = SelectKBest(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X, y)

# When to use: Target is categorical, Features are non-negative
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(score_func=chi2, k=5)
X_selected = selector.fit_transform(X, y)

#Regression (target = continuous numbers)
#only for Regression problems
from sklearn.feature_selection import SelectKBest, f_regression
selector = SelectKBest(score_func=f_regression, k=5)
X_selected = selector.fit_transform(X, y)

# convert to Dataframe (X_selected is a values array)
selected_cols = X.columns[selector.get_support()]
X_selected_df = pd.DataFrame(X_selected, columns=selected_cols)


##  **Data Splitting**

In [ ]:
X = X_selected_df

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

##  **Scaling (WHEN REQUIRED)**
-> Scaling should come after outlier handling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# Transform test data using same scaler
X_test[num_cols] = scaler.transform(X_test[num_cols])

##  **Model Training**

In [ ]:
from sklearn.metrics import accuracy_score
model = Algorithm()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(accuracy_score(y_test,y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

model = GaussianNB()

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# KNN
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# --------------------
# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression()

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("MSE:",mean_squared_error(y_test,pred))
# -------------
# Logistic Regression & SVM

# Logistic Regression

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# ----------------------------------
# SVM

from sklearn.svm import SVC

model = SVC(kernel='linear')

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# ----------------------------------------------
# PLA & MLA (Perceptron / Multi Layer)

# PLA (Perceptron Learning Algorithm)

from sklearn.linear_model import Perceptron

model = Perceptron()

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# -----------------------------------------
# MLA (MLP – Multi Layer Perceptron)

from sklearn.neural_network import MLPClassifier

model = MLPClassifier(hidden_layer_sizes=(10,10),max_iter=1000)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))

# ------------

from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=5)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

#-----------------------
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

# --------------------
# K-Means

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

model = KMeans(n_clusters=3)

model.fit(X)

pred = model.labels_

print("Silhouette Score:", silhouette_score(X, pred))
#----------------------
# DBSCAN

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

model = DBSCAN(eps=0.5, min_samples=5)

pred = model.fit_predict(X)

print("Silhouette Score:", silhouette_score(X, pred))
#-----------------
# Hierarchical (Agglomerative)

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

model = AgglomerativeClustering(n_clusters=3)

pred = model.fit_predict(X)

print("Silhouette Score:", silhouette_score(X, pred))


### **Actual vs Predicted graph**

In [ ]:
#For Regression
import matplotlib.pyplot as plt

y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred)
plt.plot(y_test, y_test)  # ideal line
plt.show()

#For Classification
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(
model, X_test, y_test,
display_labels=['Class A', 'Class B']
)

##  **PCA**
-> PCA is applied only on training data to avoid data leakage, and the same transformation is applied to test data using transform().

In [ ]:
# PCA (Dimensionality Reduction)
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)   # keeps 95% information

X_train_pca  = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

print(X_train_pca.n_components_)
print(X_test_pca.n_components_)

y_train_pca  = y_train
y_test_pca  = y_test

# Repeat model training
from sklearn.metrics import accuracy_score
model = Algorithm()

model.fit(X_train_pca,y_train_pca)

y_pred_pca = model.predict(X_test_pca)

print(accuracy_score(y_test_pca,y_pred_pca))

##  **Cross Validation (on training set)**
Splits training data into Cv(cv=5) parts. Trains multiple times. Test and Gives average accuracy

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)

print("Scores:", scores)
print("Average:", scores.mean())

#cross_val_score By default gives accuarcy score for classification task

### want all score together best way use **cross_validate**

cross_val_score() → one metric (default = accuracy)

Use scoring='...' to change metric

OR use cross_validate() to get multiple metrics together

## Anyways we can take mean for the cv score if specifically asked

## For Binary class

In [ ]:
# classification
from sklearn.model_selection import cross_validate

scores = cross_validate(model, X, y, cv=5,
                        scoring=['accuracy', 'precision', 'recall', 'f1'])

print("Accuracy:", scores['test_accuracy'])
print("Precision:", scores['test_precision'])
print("Recall:", scores['test_recall'])
print("F1:", scores['test_f1'])

# Regression
from sklearn.model_selection import cross_validate

scores = cross_validate(model, X, y, cv=5,
                        scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'])

print("R2:", scores['test_r2'].mean())
print("MSE:", -scores['test_neg_mean_squared_error'])
print("MAE:", -scores['test_neg_mean_absolute_error'])

#clustering
Clustering → silhouette (no CV)


## For MULTICLASS

micro → overall (good for imbalance)

weighted → considers class imbalance

| Type         | Meaning             | Formula                                           | When to use         |
| ------------ | ------------------- | ------------------------------------------------- | ------------------- |
| **Macro**    | All classes equal   | (\frac{P_1 + P_2 + P_3}{3})                       | Balanced data       |
| **Micro**    | Overall performance | (\frac{\sum TP}{\sum TP + \sum FP})               | Imbalanced data     |
| **Weighted** | Based on class size | (\frac{P_1 n_1 + P_2 n_2 + P_3 n_3}{n_1+n_2+n_3}) | Imbalanced (better) |


scoring = [
    'accuracy',
    'precision_macro',
    'precision_micro',
    'precision_weighted'
]

In [ ]:
from sklearn.model_selection import cross_validate

scores = cross_validate(model, X, y, cv=5,
                        scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro'])

print("Accuracy:", scores['test_accuracy'])
print("Precision:", scores['test_precision_macro'])
print("Recall:", scores['test_recall_macro'])
print("F1:", scores['test_f1_macro'])

##  **Hyperparameter Tuning**
GridSearch(Tries ALL combinations of params with cv(folds)

RandomSearch(Tries FEW random ones params with cv(folds)

Tries all parameter combinations. Finds best model automatically

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
params = {
    "max_depth": [3, 5, 10],
    "criterion": ["gini", "entropy"]
}
grid = GridSearchCV(DecisionTreeClassifier(), params, cv=5)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)


from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10]
}
random = RandomizedSearchCV(RandomForestClassifier(), params, cv=5, n_iter=4)
random.fit(X_train, y_train)
print("Best params:", random.best_params_)

**K-Fold Cross Validation:** It is a technique used to split your data into multiple parts (folds) to evaluate your model more reliably.


**Even when you pass cv=5**
scikit-learn automatically creates a KFold for you

1. If classification model → uses StratifiedKFold(Maintains class balance in each fold)

2. If regression model → uses KFold

In [ ]:
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

kf = KFold(n_splits=5)
kf = StratifiedKFold(n_splits=5)

grid = GridSearchCV(DecisionTreeClassifier(), params, cv=kf)

### only the parameter we need to study. so just study the parameter for all below.(we can see this parameter name's in the help also **ex: help(modelname)** )

In [ ]:
# ALL MODELS + IMPORTANT PARAMS
models_params = {

    # 1. KNN(same for all KNN types models)
    "KNN": {
        "model": "KNeighborsClassifier()",
        "params": {
        "n_neighbors": [3, 5, 7, 9],
        "algorithm": ["kd_tree", "ball_tree"]
        }
    },

    # 2. Naive Bayes
    "NaiveBayes": {
        "model": "GaussianNB()",
        "params": {
            # no major params needed
        }
    },

    # 3. Logistic Regression
    "LogisticRegression": {
        "model": "LogisticRegression()",
        "params": {
            "C": [0.1, 1, 10]
        }
    },

    # 4. SVM
    "SVM": {
        "model": "SVC(probability=True, random_state=42)",
        "params": {
        "kernel": ["linear", "poly", "rbf", "sigmoid"],
        "C": [0.1, 1, 10, 100],
        }
    },

    # 5. Decision Tree
    "DecisionTree": {
        "model": "DecisionTreeClassifier()",
        "params": {
            "max_depth": [3, 5, 10]
        }
    },

    # 6. Random Forest
    "RandomForest": {
        "model": "RandomForestClassifier()",
        "params": {
            "n_estimators": [50, 100],
            "max_depth": [None, 10]
        }
    },

    # 7. Neural Network (MLP)
    "NeuralNet": {
        "model": "MLPClassifier()",
        "params": {
            "hidden_layer_sizes": [(50,), (100,), (50, 30))]
        }
    },

    # 8. Ridge Regression
    "Ridge": {
        "model": "Ridge()",
        "params": {
            "alpha": [0.1, 1, 10]
        }
    },

    # 9. Lasso
    "Lasso": {
        "model": "Lasso()",
        "params": {
            "alpha": [0.1, 1, 10]
        }
    },

    # 10. ElasticNet
    "ElasticNet": {
        "model": "ElasticNet()",
        "params": {
            "alpha": [0.1, 1, 10],
            "l1_ratio": [0.5, 0.7]
        }
    },

    # 11. KMeans
    "KMeans": {
        "model": "KMeans()",
        "params": {
            "n_clusters": [2, 3, 4]
        }
    },

    # 12. DBSCAN
    "DBSCAN": {
        "model": "DBSCAN()",
        "params": {
            "eps": [0.3, 0.5, 1],
            "min_samples": [5, 10]
        }
    },

    # 13. Hierarchical Clustering
    "Hierarchical": {
        "model": "AgglomerativeClustering()",
        "params": {
            "n_clusters": [2, 3, 4]
        }
    },

    # 14. PCA
    "PCA": {
        "model": "PCA()",
        "params": {
            "n_components": [0.95]
        }
    }
}
# Load paramter
params = models_params["RandomForest"]["params"]

##  **Model Evaluation (on test set)**

In [ ]:
# Predictions
y_pred = model.predict(X_test)

In [ ]:
# Classification Metrics
# 1.Accuracy
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

# 2.Full Report(Precision,Recall,F1-score)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

# 3.Confusion Matrix
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

# 4.ROC Curve
# binary class score
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test)[:,1]
print(roc_auc_score(y_test, y_prob))

# multi class score
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test)
print(roc_auc_score(y_test, y_prob, multi_class='ovr',average=None))

# binary class curve
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.show()

# this is same as
y_prob = model.predict_proba(X_test)[:,1]
RocCurveDisplay.from_predictions(y_test, y_prob)

# multi class curve
from sklearn.metrics import RocCurveDisplay
from sklearn.preprocessing import label_binarize
# Probabilities
y_prob = model.predict_proba(X_test)

# Convert y_test to binary
y_test_bin = label_binarize(y_test, classes=model.classes_)

# Plot ROC curves
for i in range(len(model.classes_)):
    RocCurveDisplay.from_predictions(
        y_test_bin[:, i],
        y_prob[:, i],
        name=f"Class {i}",
        linestyle=['-', '=', ':'][i]
    )

plt.show()

In [ ]:
# Regression Metrics
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

##  **Model Comparison**

1. Use Cross Validation → get scores  
2. Store scores for each model  
3. Apply ANOVA or Friedman  

p < 0.05 → models significantly different

p > 0.05 → no difference

In [ ]:
from sklearn.model_selection import cross_val_score
# # models
# model1 = LogisticRegression()
# model2 = DecisionTreeClassifier()
# model3 = RandomForestClassifier()
# X → input data (columns)
# y → output (what you predict)
scores1 = cross_val_score(model1, X, y, cv=5)
scores2 = cross_val_score(model2, X, y, cv=5)
scores3 = cross_val_score(model3, X, y, cv=5)

# ANOVA
from scipy.stats import f_oneway
f, p = f_oneway(scores1, scores2, scores3)
print("ANOVA f-value, p-value:",f, p)

#ANOVA
from scipy.stats import friedmanchisquare
f, p = friedmanchisquare(scores1, scores2, scores3)
print("Friedman f-value, p-value:",f, p)

##  **Clustering**

Clustering usually has NO labels

BUT sometimes we DO have true labels (for evaluating the result ex:“Did clustering rediscover the real groups?” )

With labels → check correctness(all ready group discovered) -> ARI(adjusted_rand_score)

Without labels → discover patterns -> silhouette_score

In [ ]:
1. Load Data
2. EDA and Visualization
3. Data Preprocessing
   - Handle missing values
   - Remove duplicates
   - Handle outliers
4. Feature Selection (optional)
5. Scaling (VERY IMPORTANT)
6. (Optional) PCA
7. Apply Clustering Model
8. Evaluate (Silhouette Score)
9. Compare Models (optional)

In [ ]:
import pandas as pd
df = pd.read_csv("data.csv")

df.info()
df.describe()

df = df.dropna()

X = df.select_dtypes(include="number")

# Scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X=X_scaled

# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

# Model

# K-Means
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score


model = KMeans(n_clusters=3, random_state=42)
model.fit(X)

pred = model.labels_

print("Silhouette Score:", silhouette_score(X, pred))
print("ARI:", adjusted_rand_score(y, pred))
print("NMI:", normalized_mutual_info_score(y, pred))

# DBSCAN

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

model = DBSCAN(eps=0.5, min_samples=5)

pred = model.fit_predict(X)

print("Silhouette Score:", silhouette_score(X, pred))

# Hierarchical (Agglomerative)
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

model = AgglomerativeClustering(n_clusters=3)

pred = model.fit_predict(X)

print("Silhouette Score:", silhouette_score(X, pred))

### **K-MEANS + ELBOW METHOD**
inertia_ = WCSS (error)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# use only numeric data
X = df.select_dtypes(include="number")

wcss = []

for i in range(1, 11):
    model = KMeans(n_clusters=i)
    model.fit(X)
    wcss.append(model.inertia_)

plt.plot(range(1, 11), wcss)
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.show()

| Type           | Bagging           | Boosting           |
| -------------- | ----------------- | ------------------ |
| Classification | BaggingClassifier | AdaBoostClassifier / GradientBoostingClassifier |
| Regression     | BaggingRegressor  | AdaBoostRegressor / GradientBoostingRegressor |

### AdaBoost → focuses on misclassified points (weights)
### Gradient Boosting → focuses on reducing error using gradients

In [ ]:
# BAGGING
# Classification
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=10
)

model.fit(X_train, y_train)

# Regression
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

model = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=10
)

model.fit(X_train, y_train)

# BOOSTING
# Classification (AdaBoost)
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(n_estimators=50)

model.fit(X_train, y_train)

# Regression (AdaBoost)
from sklearn.ensemble import AdaBoostRegressor

model = AdaBoostRegressor(n_estimators=50)

model.fit(X_train, y_train)

In [ ]:
# Gradient Boosting
# Classification
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier()
model.fit(X_train, y_train)

# Regression
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor()
model.fit(X_train, y_train)

**How you want to change base model**

estimator = the base model (algorithm) used

n_estimators = number of base models

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=50
)

Ensemble works best when base models are: Simple + slightly weak + diverse

Logistic Regression : Already stable → no need bagging

SVM : Slow + complex → bad for ensembles

KNN : Already uses neighbors → bagging not useful

That why mostly Decision Trees or Random Forest are used because they are simple and combine well in ensembles learning

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

model = BaggingClassifier(estimator=KNeighborsClassifier(), n_estimators=10)

# -> Works
# -> But not popular

### **you cannot use different estimators inside one model**

One ensemble model → ONE type of estimator  
Repeated many times

Ensemble works by: Same model + different data → diversity

NOT by: Different models

## If you want different models Use: **Voting Classifier**

| Type           | Method           | Model Output  | Final Decision    | Example                                                                      |
| -------------- | ---------------- | ------------- | ----------------- | ---------------------------------------------------------------------------- |
| Classification | Hard Voting      | Class labels  | Majority vote     | LR=0, DT=1, SVM=1 → **Final = 1**                                            |
| Classification | Soft Voting      | Probabilities | Average → highest |Each model give probability for each class: <br>Class 0: (0.3+0.4+0.2)/3=0.3<br>Class 1: (0.7+0.6+0.8)/3=0.7 → **Final = 1** |
| Regression     | Voting Regressor | Numbers       | Average of predictions         | LR=100, DT=120, Another model=110 → (100+120+110)/3 = **110**                                    |


In [ ]:
# Voting Classifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model = VotingClassifier(estimators=[
    ("lr", LogisticRegression()),
    ("dt", DecisionTreeClassifier()),
    ("svm", SVC())
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))


#Soft Voting
#Soft Voting needs probabilities. if you need to use SVC()-> does NOT give
#probability → may cause issues in soft voting
#so Better version
model = VotingClassifier(estimators=[
    ("lr", LogisticRegression()),
    ("dt", DecisionTreeClassifier()),
    ("svm", SVC(probability=True))
], voting='soft')



## Voting Regressor
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

model = VotingRegressor(estimators=[
    ("lr", LinearRegression()),
    ("dt", DecisionTreeRegressor())
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))


### **Stacking**

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Base models
estimators = [
    ("lr", LogisticRegression()),
    ("dt", DecisionTreeClassifier()),
    ("svm", SVC(probability=True))  # IMPORTANT for probabilities
]

# Stacking model
model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression()
)

### Hyperparameter tuning for bagging and boosting

In [ ]:
# Bagging (Classification)
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

"BaggingClassifier": {
    "model": BaggingClassifier(estimator=DecisionTreeClassifier()),
    "params": {
        "n_estimators": [10, 50, 100],
        "max_samples": [0.5, 0.7, 1.0],
        "max_features": [0.5, 1.0],
    }

}

# Bagging (Regression)
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor

"BaggingRegressor": {
    "model": BaggingRegressor(estimator=DecisionTreeRegressor()),
    "params": {
        "n_estimators": [10, 50, 100],
        "max_samples": [0.5, 1.0],
    }
}

# AdaBoost (Classification)
from sklearn.ensemble import AdaBoostClassifier

"AdaBoostClassifier": {
    "model": AdaBoostClassifier(),
    "params": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 1]
    }
}

# AdaBoost (Regression)
from sklearn.ensemble import AdaBoostRegressor

"AdaBoostRegressor": {
    "model": AdaBoostRegressor(),
    "params": {
        "n_estimators": [50, 100],
        "learning_rate": [0.01, 0.1, 1]
    }
},